# Inference Pipeline

# DISTILLBERT

In [1]:
import os
import json
import torch
from transformers import AutoModelForTokenClassification, AutoTokenizer
import sys

# Add parent directory to sys.path so we can import from the project root (useful for Jupyter or script)
# os.getcwd()        → returns current working directory, e.g., "/path/to/symptom-ner/v01"
# os.path.join(..., "..") → moves one directory up, i.e., "/path/to/symptom-ner"
# os.path.abspath()  → resolves this to the absolute path
PARENT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PARENT_DIR not in sys.path:
    sys.path.insert(0, PARENT_DIR)

from gcp_utils import download_from_gcs, list_bucket_files
from config import settings


# ------- Load labels and test data - LOCALLY -------
with open("data/distillbert_splits/test.jsonl", "r") as f:
    test_data = []
    for line in f:
        test_data.append(line)

with open("data/id2label.json", "r") as f:
    id2label = json.load(f)
with open("data/label2id.json", "r") as f:
    label2id = json.load(f)
# ------------------------------------------------------

# Convert id2label keys from strings to integers (JSON loads keys as strings)
if any(isinstance(k, str) for k in id2label.keys()):
    id2label = {int(k): v for k, v in id2label.items()}


# CONFIG FOR LOADING FROM GCS 

# v01/runs/distilbert-base-uncased/run_0/
VERSION = "v01"
MODEL_NAME = "distilbert-base-uncased"  # or "dmis-lab/biobert-base-cased-v1.2" for BioBERT
RUN_IDX = 0  

GCS_MODEL_PATH = f"{VERSION}/runs/{MODEL_NAME}/run_{RUN_IDX}"
BUCKET_NAME = settings.BUCKET_NAME  # "ner_training_data_results"


/Users/robertagarcia/Desktop/learning/bert_symptom_ner/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Create a local directory to download the model
LOCAL_MODEL_DIR = f"./downloaded_models/{MODEL_NAME}/run_{RUN_IDX}"

# Download the model directory from GCS
print(f"Downloading model from gs://{BUCKET_NAME}/{GCS_MODEL_PATH}...")
downloaded_path = download_from_gcs(
    gcs_path=GCS_MODEL_PATH,
    local_path=LOCAL_MODEL_DIR,
    bucket_name=BUCKET_NAME
)

# Load the model
model = AutoModelForTokenClassification.from_pretrained(LOCAL_MODEL_DIR)
tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_DIR)

# Move to device
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
    
print(f"Using device: {device}")
model.to(device)
model.eval()

✅  Downloaded all files from the folder
✓ Downloaded directory: gs://ner_training_data_results/v01/runs/distilbert-base-uncased/run_0 (24 files) → ./downloaded_models/distilbert-base-uncased/run_0
Using device: mps


DistilBertForTokenClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
   

# **Token Level Prediction**

In [3]:
from inference_utils import predict_token_level
test_text = "Patient reports severe headache and nausea"
tokens, predictions = predict_token_level(test_text, model, tokenizer, device=device)
print(f"Tokens {len(tokens)}:\n\t{tokens}")
print(f"Predictions {len(predictions)}:\n\t{predictions}")

Tokens 6:
	['patient', 'reports', 'severe', 'headache', 'and', 'nausea']
Predictions 6:
	[4, 4, 1, 3, 3, 3]


In [4]:
# RUN ANOTHER EXAMPLE: 
sample = json.loads(test_data[1])
text = sample.get('text')
tokens = sample.get('tokens')
token_label_ids = sample.get('token_label_ids')
print(f"TEXT: {text}")
print(f"TOKENS from test data: {tokens}")
tks, predictions = predict_token_level(text, model, tokenizer, device=device)
print(f"Returned tokens: {tks}")
print(f"Predictions: {predictions[0]}")


TEXT: The patient has lymphatic system symptom.
TOKENS from test data: ['[CLS]', 'the', 'patient', 'has', 'l', '##ym', '##pha', '##tic', 'system', 'sy', '##mpt', '##om', '.', '[SEP]']
Returned tokens: ['the', 'patient', 'has', 'l', '##ym', '##pha', '##tic', 'system', 'sy', '##mpt', '##om', '.']
Predictions: 4


# **Word Level Prediction**

In [3]:
from inference_utils import predict_word_level

sample = json.loads(test_data[1])
text = sample.get('text')

predict_word_level(text=text, model=model, tokenizer=tokenizer, id2label=id2label, device=device)

(['the',
  'patient',
  'has',
  'l',
  '##ym',
  '##pha',
  '##tic',
  'system',
  'sy',
  '##mpt',
  '##om',
  '.'],
 ['O',
  'O',
  'O',
  'B-SYMPTOM_POS',
  'I-SYMPTOM_POS',
  'I-SYMPTOM_POS',
  'I-SYMPTOM_POS',
  'I-SYMPTOM_POS',
  'I-SYMPTOM_POS',
  'I-SYMPTOM_POS',
  'I-SYMPTOM_POS',
  'O'],
 [0, 1, 2, 3, 3, 3, 3, 4, 5, 5, 5, 6],
 ['The', 'patient', 'has', 'lymphatic', 'system', 'symptom.'],
 ['O',
  'O',
  'O',
  'CONFLICT-B-SYMPTOM_POS-I-SYMPTOM_POS-I-SYMPTOM_POS-I-SYMPTOM_POS',
  ['I-SYMPTOM_POS'],
  ['I-SYMPTOM_POS', 'I-SYMPTOM_POS', 'I-SYMPTOM_POS'],
  'O'])

'{"text": "The patient has lymphatic system symptom.", "word_tokens": ["The", "patient", "has", "lymphatic", "system", "symptom", "."], "word_labels": ["O", "O", "O", "B-SYMPTOM_POS", "I-SYMPTOM_POS", "I-SYMPTOM_POS", "O"], "tokens": ["[CLS]", "the", "patient", "has", "l", "##ym", "##pha", "##tic", "system", "sy", "##mpt", "##om", ".", "[SEP]"], "input_ids": [101, 1996, 5776, 2038, 1048, 24335, 21890, 4588, 2291, 25353, 27718, 5358, 1012, 102], "token_labels": ["None", "O", "O", "O", "B-SYMPTOM_POS", "I-SYMPTOM_POS", "I-SYMPTOM_POS", "I-SYMPTOM_POS", "I-SYMPTOM_POS", "I-SYMPTOM_POS", "I-SYMPTOM_POS", "I-SYMPTOM_POS", "O", "None"], "token_label_ids": [-100, 4, 4, 4, 1, 3, 3, 3, 3, 3, 3, 3, 4, -100]}\n'

In [9]:
from inference_utils import predict_word_level

#sample = json.loads(test_data[1])
text = "The patient has cataplexy." #lymphatic system symptom and cataplexy."#sample.get('text')
samples = ["The patient has cataplexy.", "The patient has lymphatic system symptom.", "The patient has lymphatic system symptom and cataplexy.","The patient has lymphatic system symptom.", "The patient has cataplexy and lymphatic system symptom."]
for s in samples:
    print("="*20)
    print(f" TEXT: {s}")
    print("="*20)
    tokens,token_labels,word_ids, words, word_labels =  predict_word_level(text=s, model=model, tokenizer=tokenizer, id2label=id2label, device=device)
    print("tokens: ", tokens)
    print("token labels: ", token_labels)
    print("word_ids: ", word_ids)
    print("word: ", words)
    print("bio word labels: ", word_labels)
    print()

 TEXT: The patient has cataplexy.
tokens:  ['the', 'patient', 'has', 'cat', '##ap', '##le', '##xy', '.']
token labels:  ['O', 'O', 'O', 'B-SYMPTOM_POS', 'I-SYMPTOM_POS', 'I-SYMPTOM_POS', 'I-SYMPTOM_POS', 'O']
word_ids:  [0, 1, 2, 3, 3, 3, 3, 4]
word:  ['The', 'patient', 'has', 'cataplexy.']
bio word labels:  ['O', 'O', 'O', 'B-SYMPTOM_POS', 'O']

 TEXT: The patient has lymphatic system symptom.
tokens:  ['the', 'patient', 'has', 'l', '##ym', '##pha', '##tic', 'system', 'sy', '##mpt', '##om', '.']
token labels:  ['O', 'O', 'O', 'B-SYMPTOM_POS', 'I-SYMPTOM_POS', 'I-SYMPTOM_POS', 'I-SYMPTOM_POS', 'I-SYMPTOM_POS', 'I-SYMPTOM_POS', 'I-SYMPTOM_POS', 'I-SYMPTOM_POS', 'O']
word_ids:  [0, 1, 2, 3, 3, 3, 3, 4, 5, 5, 5, 6]
word:  ['The', 'patient', 'has', 'lymphatic', 'system', 'symptom.']
bio word labels:  ['O', 'O', 'O', 'B-SYMPTOM_POS', 'I-SYMPTOM_POS', 'I-SYMPTOM_POS', 'O']

 TEXT: The patient has lymphatic system symptom and cataplexy.
tokens:  ['the', 'patient', 'has', 'l', '##ym', '##pha',